# PaliGemma 2: Türkçe OCR, 224 vs 448
Soru: çözünürlük artışı (224 → 448) Türkçe tabela/fiş okumada (ş ğ ı ç ö ü) fark yaratıyor mu?
Koşullar: 2 çözünürlük × 3 prompt (`ocr`, `answer tr ...`, `answer en ...`). Çıktılar CSV'ye gider, doğru/kısmen/yanlış **elle** işaretlenir.

Çalışma zamanı: GPU (T4). Önce HF'de `google/paligemma2-3b-mix-224` ve `-448` lisanslarını kabul et.

In [1]:
!pip -q install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 141.5 MB/s eta 0:00:00


In [3]:
from huggingface_hub import notebook_login
notebook_login()

In [11]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
!ls -l "/content/drive/MyDrive/fotolar"

Mounted at /content/drive
total 33045
-rw------- 1 root root 16901518 Oct  4 15:02 Archive.zip
-rw------- 1 root root  1059454 Oct  4 14:44 fis_01.jpg
-rw------- 1 root root   874386 Oct  4 14:45 fis_02.jpg
-rw------- 1 root root   732770 Oct  4 14:45 fis_03.jpg
-rw------- 1 root root   901537 Oct  4 14:45 fis_04.jpg
-rw------- 1 root root  1112896 Oct  4 14:45 fis_05.jpg
-rw------- 1 root root   556832 Oct  4 14:45 fis_06.jpg
-rw------- 1 root root  1078661 Oct  4 14:45 fis_07.jpg
-rw------- 1 root root  2359047 Oct  4 14:50 kutu_01.jpg
-rw------- 1 root root  2285386 Oct  4 14:50 kutu_02.jpg
-rw------- 1 root root  2270587 Oct  4 14:50 kutu_03.jpg
-rw------- 1 root root  3702112 Oct  4 14:50 tabela_01.jpg


In [6]:
!unzip -o Archive.zip -d fotolar

Archive:  Archive.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of Archive.zip or
        Archive.zip.zip, and cannot find Archive.zip.ZIP, period.


In [ ]:
!ls -l fotolar/
!file fotolar/kutu_01.jpg
!head -c 16 fotolar/kutu_01.jpg | xxd

total 9508
-rw-r--r-- 1 root root 1059454 Oct  4 14:52 fis_01.jpg
-rw-r--r-- 1 root root  874386 Oct  4 14:51 fis_02.jpg
-rw-r--r-- 1 root root  732770 Oct  4 14:51 fis_03.jpg
-rw-r--r-- 1 root root  901537 Oct  4 14:51 fis_04.jpg
-rw-r--r-- 1 root root 1112896 Oct  4 14:51 fis_05.jpg
-rw-r--r-- 1 root root  556832 Oct  4 14:52 fis_06.jpg
-rw-r--r-- 1 root root 1078661 Oct  4 14:52 fis_07.jpg
-rw-r--r-- 1 root root  261895 Oct  4 14:52 kutu_01.jpg
-rw-r--r-- 1 root root 1048576 Oct  4 14:52 kutu_02.jpg
-rw-r--r-- 1 root root 1048576 Oct  4 14:52 kutu_03.jpg
-rw-r--r-- 1 root root 1048576 Oct  4 14:52 tabela_01.jpg
fotolar/kutu_01.jpg: data
/bin/bash: line 1: xxd: command not found


## Fotoğrafları yükle
Sol panel → Files → `fotolar/` klasörü aç → 15–20 görseli sürükle. Dosya adı = örnek adı (ör. `tabela_01.jpg`).

In [12]:
import os, glob, gc, torch, pandas as pd
from PIL import Image
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration

os.makedirs("fotolar", exist_ok=True)
paths = sorted(p for p in glob.glob("/content/drive/MyDrive/fotolar/*") if p.lower().endswith((".jpg",".jpeg",".png",".webp")))
print(len(paths), "görsel")

MODELS = {224: "google/paligemma2-3b-mix-224", 448: "google/paligemma2-3b-mix-448"}
PROMPTS = {
    "ocr": "ocr",
    "tr":  "answer tr Görselde ne yazıyor?",
    "en":  "answer en What does the text in the image say?",
}
# T4 bf16'yı donanımda desteklemez; çalışır ama yavaş olabilir. Hata verirse torch.float16 dene
# (Gemma 2'de fp16 taşma yapabilir, çıktı bozuksa bf16'ya dön).
DTYPE = torch.bfloat16

11 görsel


In [13]:
rows = []
for res, mid in MODELS.items():
    model = PaliGemmaForConditionalGeneration.from_pretrained(mid, torch_dtype=DTYPE, device_map="auto").eval()
    proc = PaliGemmaProcessor.from_pretrained(mid)
    for p in paths:
        img = Image.open(p).convert("RGB")
        for pname, prompt in PROMPTS.items():
            inp = proc(text=prompt, images=img, return_tensors="pt").to(DTYPE).to(model.device)
            n = inp["input_ids"].shape[-1]
            with torch.inference_mode():
                out = model.generate(**inp, max_new_tokens=100, do_sample=False)
            text = proc.decode(out[0][n:], skip_special_tokens=True)
            rows.append(dict(foto=os.path.basename(p), cozunurluk=res, prompt=pname, cikti=text))
    del model, proc; gc.collect(); torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df["dogru_kismen_yanlis"] = ""   # elle: d / k / y
df["turkce_karakter_ok"] = ""    # elle: e / h
df["gercek_metin"] = ""          # elle: fotoğraftaki asıl yazı
df.to_csv("sonuclar.csv", index=False)
df.head(12)

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/75.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.6MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many i

,foto,cozunurluk,prompt,cikti,dogru_kismen_yanlis,turkce_karakter_ok,gercek_metin
0,fis_01.jpg,224,ocr,K.S.\nK.\nENVER BABA GÜRME GIDA\nSAN.T.C. LTD....,,,
1,fis_01.jpg,224,tr,"Bir mağazanın faturası, içinde bir numara ve p...",,,
2,fis_01.jpg,224,en,ENVER BABA GÜRME GIDA SAN.T.C. LTD.ŞTİ.\n: 01/...,,,
3,fis_02.jpg,224,ocr,EVA DONDURMA\nMERKEZ ŞARTALİ\nYILDIZTABYA MAHA...,,,
4,fis_02.jpg,224,tr,"evrakta ""eva dondurma"" yazıyor.",,,
5,fis_02.jpg,224,en,eva dondurma,,,
6,fis_03.jpg,224,ocr,İBB İŞLETMELER MÜDÜRLÜĞÜ\nİSTANBUL İL BAŞKANLI...,,,
7,fis_03.jpg,224,tr,Birisi bir mağazanın faturasını tutuyor.,,,
8,fis_03.jpg,224,en,İBB İŞLETLERİLER MÜDÜRLÜĞÜ 02.01.2021 16:00-18...,,,
9,fis_04.jpg,224,ocr,KEOBS GIDA LTD STİ ALIBEYKOY PH 4. GÜL SK NO:9...,,,


## Elle işaretleme
`sonuclar.csv`'yi indir, Sheets'te aç, `gercek_metin` + `dogru_kismen_yanlis` + `turkce_karakter_ok` sütunlarını doldur. Sonra özet tablo için:

In [ ]:
df = pd.read_csv("sonuclar.csv")  # işaretli halini yükle
print(pd.crosstab([df.prompt, df.cozunurluk], df.dogru_kismen_yanlis))
print(pd.crosstab([df.prompt, df.cozunurluk], df.turkce_karakter_ok))